In [ ]:
!pip install OpenDartReader
!pip install dart-fss
!pip install pandas requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.1/148.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 7.9 MB/s eta 0:00:00


In [ ]:
# 필요 import 호출
import pandas as pd
import requests
import zipfile
import io
from tqdm import tqdm
import math
import time


In [ ]:
DART_KEY = '4d73cf54c2edf5512ba3d5b98feb95aa729b8cc9'

url = f"https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={DART_KEY}"
response = requests.get(url) # 해당 URL로 HTTP GET 요청을 보내고 결과를 가져오기

with zipfile.ZipFile(io.BytesIO(response.content)) as z: # ZIP 파일 데이터를 바이트 형태로 가지고온 후 사용하면 메모리 상에서 파일처럼 다룰 수 있다.
    z.extractall()

In [ ]:
# 전체 기업 갯수
corp_df = pd.read_xml("CORPCODE.xml")
corp_df.head()

len(corp_df)

115066

In [ ]:
# 그 중에 상장된 기업 갯수
# listed_df는 주식 코드가 없거나 공백일 경우 제거한다.
listed_df = corp_df[
    (corp_df["stock_code"].notna()) &
    (corp_df["stock_code"] != " ")
].copy()

listed_df.shape

(3945, 5)

In [ ]:
ACCOUNT_MAP = {
    "자산총계": ["자산총계", "총자산"],
    "부채총계": ["부채총계", "총부채"],
    "자본총계": ["자본총계", "자본"],
    "자본금": ["자본금"],
    "유동자산": ["유동자산"],
    "비유동자산": ["비유동자산"],
    "유동부채": ["유동부채"],
    "비유동부채": ["비유동부채"],
    "재고자산": ["재고자산"],
    "이익잉여금": ["이익잉여금", "이익준비금"],
    "매출액": ["매출액", "매출", "영업수익"],
    "영업이익": ["영업이익"],
    "당기순이익": ["당기순이익", "당기순손익"],
    "매출원가": ["매출원가"],
    "영업활동현금흐름": ["영업활동으로인한현금흐름"],
    "투자활동현금흐름": ["투자활동으로인한현금흐름"],
    "재무활동현금흐름": ["재무활동으로인한현금흐름"]
}

In [ ]:
def get_financial_safe(corp_code, year):
    reprt_codes = ["11011", "11012", "11013", "11014"]  # 1~4분기
    for code in reprt_codes:
        for attempt in range(2):  # 재시도 1회
            try:
                resp = requests.get(
                    "https://opendart.fss.or.kr/api/fnlttSinglAcnt.json",
                    params={
                        "crtfc_key": DART_KEY,
                        "corp_code": corp_code,
                        "bsns_year": year,
                        "reprt_code": code
                    },
                    timeout=10
                ).json()

                if resp.get("status") == "000" and resp.get("list"):
                    return pd.DataFrame(resp["list"])
                # status != "000"이면 그냥 다음 분기 코드로
                break  # 재시도 끝나면 다음 코드로
            except requests.exceptions.RequestException as e:
                print(f"{corp_code} {year} 요청 실패: {e}, 재시도 {attempt+1}")
                time.sleep(1)
    return pd.DataFrame()



In [ ]:
# 재무제표 데이터에서 원하는 금액, 즉 입력값 x를 숫자(float)로 변환해주는 함수
def to_num_safe(x):
    try:
        if isinstance(x, str):
            x_clean = x.replace(",", "").strip()
            if x_clean in ["", "-"]:
                return None
            return float(x_clean)
        elif isinstance(x, (int, float)):
            return float(x)
    except ValueError:
        return None
    return None

# 재무제표 DataFrame df에서 특정 항목(keys)에 해당하는 금액을 찾아 숫자로 반환, 없으면 None 반환.
def pick_amount_flex_safe(df, keys):
    if df is None or df.empty:
        return None
    mask = df["account_nm"].notna() & df["account_nm"].apply(lambda x: any(k in x for k in keys))
    row = df[mask]
    if not row.empty and "thstrm_amount" in row.columns:
        return to_num_safe(row.iloc[0]["thstrm_amount"])
    return None



In [ ]:
def calc_ratios_loose_safe(fs_cur, fs_prev):
    # 기본 항목
    asset_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["자산총계"])
    asset_prev = pick_amount_flex_safe(fs_prev, ACCOUNT_MAP["자산총계"]) if fs_prev is not None else None

    debt_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["부채총계"])
    equity_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["자본총계"])
    equity_prev = pick_amount_flex_safe(fs_prev, ACCOUNT_MAP["자본총계"]) if fs_prev is not None else None

    sales_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["매출액"])
    sales_prev = pick_amount_flex_safe(fs_prev, ACCOUNT_MAP["매출액"]) if fs_prev is not None else None

    net_income_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["당기순이익"])
    net_income_prev = pick_amount_flex_safe(fs_prev, ACCOUNT_MAP["당기순이익"]) if fs_prev is not None else None

    op_income_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["영업이익"])
    op_income_prev = pick_amount_flex_safe(fs_prev, ACCOUNT_MAP["영업이익"]) if fs_prev is not None else None

    # 추가 항목
    current_assets_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["유동자산"])
    current_liab_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["유동부채"])
    noncurrent_assets_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["비유동자산"])
    noncurrent_liab_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["비유동부채"])
    inventory_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["재고자산"])
    retained_earnings_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["이익잉여금"])
    capital_stock_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["자본금"])
    cogs_cur = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["매출원가"])
    cf_oper = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["영업활동현금흐름"])
    cf_inv = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["투자활동현금흐름"])
    cf_fin = pick_amount_flex_safe(fs_cur, ACCOUNT_MAP["재무활동현금흐름"])

    ratios = {
        "ROA": (net_income_cur / asset_cur) if asset_cur not in [None, 0] and net_income_cur is not None else None,
        "ROE": (net_income_cur / equity_cur) if equity_cur not in [None, 0] and net_income_cur is not None else None,
        "순이익률": (net_income_cur / sales_cur) if sales_cur not in [None,0] and net_income_cur is not None else None,
        "매출액증가율": ((sales_cur - sales_prev) / sales_prev) if sales_prev not in [None,0] and sales_cur is not None else None,
        "자산증가율": ((asset_cur - asset_prev) / asset_prev) if asset_prev not in [None,0] and asset_cur is not None else None,
        "영업이익증가율": ((op_income_cur - op_income_prev) / op_income_prev) if op_income_prev not in [None,0] and op_income_cur is not None else None,
        "순이익증가율": ((net_income_cur - net_income_prev) / net_income_prev) if net_income_prev not in [None,0] and net_income_cur is not None else None,
        "영업이익률": (op_income_cur / sales_cur) if sales_cur not in [None,0] and op_income_cur is not None else None,
        "부채비율": (debt_cur / equity_cur * 100) if equity_cur not in [None,0] and debt_cur is not None else None,
        "자기자본비율": (equity_cur / asset_cur * 100) if asset_cur not in [None,0] and equity_cur is not None else None,
        "유동비율": (current_assets_cur / current_liab_cur * 100) if current_liab_cur not in [None,0] and current_assets_cur is not None else None,
        "총자산회전율": (sales_cur / asset_cur) if asset_cur not in [None,0] and sales_cur is not None else None,
        "재고자산회전율": (cogs_cur / inventory_cur) if inventory_cur not in [None,0] and cogs_cur not in [None,0] else None,
        "영업활동현금흐름": cf_oper,
        "투자활동현금흐름": cf_inv,
        "재무활동현금흐름": cf_fin,
        "총자산": asset_cur,
        "유동자산": current_assets_cur,
        "비유동자산": noncurrent_assets_cur,
        "유동부채": current_liab_cur,
        "비유동부채": noncurrent_liab_cur,
        "총부채": debt_cur,
        "자본금": capital_stock_cur,
        "자본총계": equity_cur,
        "이익잉여금": retained_earnings_cur
    }
    return ratios


In [ ]:
YEARS = list(range(2015, 2026))

# 7. 삼성전자 테스트
samsung = {
    "corp_code": "00126380",
    "stock_code": "005930",
    "corp_name": "삼성전자"
}

rows = []
prev_fs = None

for year in YEARS:
    cur_fs = get_financial_safe(samsung["corp_code"], year)

    if cur_fs.empty:
        prev_fs = None
        continue

    ratios = calc_ratios_loose_safe(cur_fs, prev_fs)

    if year >= 2016 and any(v is not None for v in ratios.values()):
        rows.append({
            "종목코드": samsung["stock_code"],
            "기업명": samsung["corp_name"],
            "분석연도": year,
            **ratios
        })

    prev_fs = cur_fs

dart_df = pd.DataFrame(rows)

# 8. Excel 저장
dart_df.to_excel("삼성전자_financial_ratios2.xlsx", index=False)

print("✅ 저장 완료")
print(dart_df)

✅ 저장 완료
     종목코드   기업명  분석연도       ROA        ROE      순이익률    매출액증가율     자산증가율  \
0  005930  삼성전자  2016  0.086683   0.117774  0.112580  0.006047  0.082562   
1  005930  삼성전자  2017  0.139806  47.003999  0.176090  0.186800  0.150960   
2  005930  삼성전자  2018  0.130673  49.408541  0.181912  0.017514  0.124623   
3  005930  삼성전자  2019  0.061659  24.221199  0.094352 -0.054849  0.038918   
4  005930  삼성전자  2020  0.069818  29.423309  0.111516  0.027804  0.072813   
5  005930  삼성전자  2021  0.093543  44.464432  0.142728  0.180729  0.127924   
6  005930  삼성전자  2022  0.124110  62.009146  0.184144  0.080923  0.051107   
7  005930  삼성전자  2023  0.033970  17.255553  0.059811 -0.143254  0.016684   
8  005930  삼성전자  2024  0.066957  38.385308  0.114505  0.161953  0.128592   
9  005930  삼성전자  2025  0.010134   5.700674  0.068616 -0.752165 -0.018768   

    영업이익증가율    순이익증가율  ...  재무활동현금흐름           총자산          유동자산  \
0  0.107038  0.192336  ...      None  2.621743e+14  1.414297e+14   
1  0.834603  0.8563

### 여기선 종목별 코드를 분류할 예정입니다.

In [ ]:
## 있는지 여부 확인
fs = get_financial_safe("00126380", 2025)  # 삼성전자 실제 corp_code 예시
print(fs.head())

         rcept_no reprt_code bsns_year corp_code stock_code fs_div   fs_nm  \
0  20250814003325      11012      2025  00307189     057500    CFS  연결재무제표   
1  20250814003325      11012      2025  00307189     057500    CFS  연결재무제표   
2  20250814003325      11012      2025  00307189     057500    CFS  연결재무제표   
3  20250814003325      11012      2025  00307189     057500    CFS  연결재무제표   
4  20250814003325      11012      2025  00307189     057500    CFS  연결재무제표   

  sj_div  sj_nm account_nm  thstrm_nm      thstrm_dt    thstrm_amount  \
0     BS  재무상태표       유동자산  제 31 기반기말  2025.06.30 현재  396,992,252,086   
1     BS  재무상태표      비유동자산  제 31 기반기말  2025.06.30 현재   84,666,504,168   
2     BS  재무상태표       자산총계  제 31 기반기말  2025.06.30 현재  481,658,756,254   
3     BS  재무상태표       유동부채  제 31 기반기말  2025.06.30 현재   84,491,356,822   
4     BS  재무상태표      비유동부채  제 31 기반기말  2025.06.30 현재    1,939,775,960   

  frmtrm_nm      frmtrm_dt    frmtrm_amount ord currency thstrm_add_amount  \
0   제 30 기말  2

In [ ]:
# 6. 분석 연도
YEARS = list(range(2016, 2026))  # 10년

# 7. 삼성전자 테스트
samsung = {
    "corp_code": "00307189",
    "stock_code": "057500",
    "corp_name": "에스케이엔펄스"
}

rows = []
prev_fs = None
for year in YEARS:
    cur_fs = get_financial_safe(samsung["corp_code"], year)
    if cur_fs.empty:
        prev_fs = None
        continue

    ratios = calc_ratios_loose_safe(cur_fs, prev_fs)
    if any(v is not None for v in ratios.values()):
        rows.append({
            "종목코드": samsung["stock_code"],
            "기업명": samsung["corp_name"],
            "분석연도": year,
            **ratios
        })

    prev_fs = cur_fs

dart_df = pd.DataFrame(rows)

# 컬럼 순서
cols = ["종목코드", "기업명", "분석연도",
        "ROA","ROE","순이익률","매출액증가율","자산증가율","영업이익증가율","순이익증가율",
        "영업이익률","부채비율","자기자본비율","유동비율","총자산회전율","재고자산회전율",
        "영업활동현금흐름","투자활동현금흐름","재무활동현금흐름",
        "총자산","유동자산","비유동자산","유동부채","비유동부채","총부채","자본금","자본총계","이익잉여금"]
dart_df = dart_df.reindex(columns=cols)

# =====================================
# 8. Excel 저장
# =====================================
dart_df.to_excel("에스케이엔펄스_financial_ratios_extended.xlsx", index=False)

print("✅ 저장 완료")
print(dart_df)

✅ 저장 완료
     종목코드      기업명  분석연도       ROA       ROE        순이익률    매출액증가율     자산증가율  \
0  057500  에스케이엔펄스  2016 -0.230835 -1.158470   -0.376998       NaN       NaN   
1  057500  에스케이엔펄스  2017  0.234603  1.282192    0.297083  0.404525  0.089021   
2  057500  에스케이엔펄스  2018  0.120700  0.797968    0.174907  0.057071  0.209647   
3  057500  에스케이엔펄스  2019  0.001732  0.012959    0.002870 -0.010184  0.131796   
4  057500  에스케이엔펄스  2020  0.053407  0.448664    0.078928  0.258802  0.122726   
5  057500  에스케이엔펄스  2021  0.013719  0.094655    0.024810  0.554581  0.902341   
6  057500  에스케이엔펄스  2022  0.019461  0.141887    0.033200  0.120180  0.056732   
7  057500  에스케이엔펄스  2023  0.017207  0.101792    0.104519 -0.718183  0.003431   
8  057500  에스케이엔펄스  2024  0.500552  2.538245    4.466008 -0.592547 -0.401526   
9  057500  에스케이엔펄스  2025  0.412710  3.232655  110.385413 -0.948473  0.544649   

     영업이익증가율     순이익증가율  ...  재무활동현금흐름           총자산          유동자산  \
0        NaN        NaN  ...      None  1

## 이 위로는 테스트용

In [1]:
# 1. 엑셀에서 상위 100개 기업 불러오기

corp_list = pd.read_csv("corp.csv").iloc[:150]


# 2. 분석 연도
#    - 2015: 계산용
#    - 2016~: 출력용

YEARS = list(range(2015, 2026))

# 3. 기업 반복 분석
rows = []

for _, row in corp_list.iterrows():
    corp_code = str(row["corp_code"]).zfill(8)
    stock_code = row["stock_code"]
    corp_name = row["corp_name"]
    prev_fs = None

    for year in YEARS:
        cur_fs = get_financial_safe(corp_code, year)

        if cur_fs.empty:
            prev_fs = None
            continue

        ratios = calc_ratios_loose_safe(cur_fs, prev_fs)

        # ✅ 2016년부터만 결과 저장
        if year >= 2016 and any(v is not None for v in ratios.values()):
            rows.append({
                "종목코드": stock_code,
                "기업명": corp_name,
                "분석연도": year,
                **ratios
            })

        prev_fs = cur_fs

# =====================================
# 4. DataFrame 생성 + 컬럼 순서
# =====================================
cols = ["종목코드","기업명","분석연도",
        "ROA","ROE","순이익률","매출액증가율","자산증가율","영업이익증가율","순이익증가율",
        "영업이익률","부채비율","자기자본비율","유동비율","총자산회전율","재고자산회전율",
        "영업활동현금흐름","투자활동현금흐름","재무활동현금흐름",
        "총자산","유동자산","비유동자산","유동부채","비유동부채","총부채","자본금","자본총계","이익잉여금"]

dart_df_test = pd.DataFrame(rows)
dart_df_test = dart_df_test.reindex(columns=cols)

# =====================================
# 5. Excel 저장
# =====================================
dart_df_test.to_excel("top10_financial_ratios26.xlsx", index=False)

print("✅ 상위 기업 분석 결과 엑셀 저장 완료!")
print(dart_df_test.head())

FileNotFoundError: [Errno 2] No such file or directory: 'corp.csv'

## 폐기 코드
### 이유
 - 재활용 여지는 있다.
 - 분석하는데 오류 발생 및 기존 코드에서의 결합 문제 발생

In [ ]:
# 1. 엑셀에서 상위 100개 기업 불러오기
corp_list = pd.read_csv("corp.csv").iloc[:100]  # 상위 100개만

# 2. 분석 연도
YEARS = list(range(2016, 2026))  # 10년


# 3. 100개 기업 반복 테스트

rows = []

for _, row in corp_list.iterrows():
    corp_code = str(row["corp_code"]).zfill(8)
    stock_code = row["stock_code"]
    corp_name = row["corp_name"]
    prev_fs = None

    for year in YEARS:
        cur_fs = get_financial_safe(corp_code, year)
        if cur_fs.empty:
            prev_fs = None
            continue

        ratios = calc_ratios_loose_safe(cur_fs, prev_fs)

        if any(v is not None for v in ratios.values()):
            rows.append({
                "종목코드": stock_code,
                "기업명": corp_name,
                "분석연도": year,
                **ratios
            })

        prev_fs = cur_fs

# 4. DataFrame 생성 + 컬럼 순서
cols = ["종목코드","기업명","분석연도",
        "ROA","ROE","순이익률","매출액증가율","자산증가율","영업이익증가율","순이익증가율",
        "영업이익률","부채비율","자기자본비율","유동비율","총자산회전율","재고자산회전율",
        "영업활동현금흐름","투자활동현금흐름","재무활동현금흐름",
        "총자산","유동자산","비유동자산","유동부채","비유동부채","총부채","자본금","자본총계","이익잉여금"]

dart_df_test = pd.DataFrame(rows)
dart_df_test = dart_df_test.reindex(columns=cols)

# 5. Excel 저장
dart_df_test.to_excel("top10_financial_ratios13.xlsx", index=False) ## 14부터 진행
print("@상위 100개 기업 분석 결과 엑셀 저장 완료!")
print(dart_df_test.head())

✅ 상위 10개 기업 분석 결과 엑셀 저장 완료!
    종목코드   기업명  분석연도       ROA       ROE      순이익률    매출액증가율     자산증가율  \
0  94860  네오리진  2016  0.043255  0.057123  0.063315       NaN       NaN   
1  94860  네오리진  2017  0.009399  0.034879  0.017942 -0.130574  0.133862   
2  94860  네오리진  2018 -0.257378 -0.793506 -0.417156 -0.021474 -0.169195   
3  94860  네오리진  2019 -0.193830 -0.538393 -0.241926  0.243345 -0.042532   
4  94860  네오리진  2020 -0.157485 -0.355620 -0.190110 -0.159450 -0.187038   

    영업이익증가율     순이익증가율  ...  재무활동현금흐름           총자산          유동자산  \
0       NaN        NaN  ...      None  1.559592e+10  6.580263e+09   
1 -0.791961  -0.753621  ...      None  1.768362e+10  7.853986e+09   
2 -5.574660 -23.750553  ...      None  1.469163e+10  6.149983e+09   
3 -1.334725  -0.278934  ...      None  1.406676e+10  7.948947e+09   
4 -7.585675  -0.339479  ...      None  1.143575e+10  5.251876e+09   

          비유동자산          유동부채         비유동부채           총부채           자본금  \
0  9.015657e+09  2.777733e+09  1.0086